# 5a — The Corridor Problem

Point-to-point ICP (Module 4) works well in a cluttered room. Put the robot
in a long, featureless hallway and it breaks in a very specific way: it
happily reports that the robot has *not moved along the corridor*, no matter
how far it actually drove. This notebook reproduces that failure and looks
at why it happens.

**Learning goals**

1. Run a compact point-to-point ICP on a corridor and watch it under-estimate forward motion.
2. See that the cost is flat along the corridor: that direction is *unobservable* from the walls.
3. See that a single feature restores observability.
4. Understand what a point-to-*line* metric changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

## Setup: simulator and a compact ICP

The LiDAR simulator from [1a](1a_polar_to_cartesian.ipynb) and a minimal
point-to-point ICP (nearest neighbours + the closed-form SVD solution) are
repeated here. ICP itself is the subject of Module 4; the version below is
deliberately short.

In [ ]:
def cast_ray(origin, angle, segments, max_range):
    """Distance along a ray from `origin` at `angle` to the nearest wall segment.
    Returns `max_range` if nothing is hit."""
    d = np.array([np.cos(angle), np.sin(angle)])
    best = max_range
    for p, q in segments:
        p, q = np.asarray(p, float), np.asarray(q, float)
        e = q - p
        denom = d[0] * e[1] - d[1] * e[0]
        if abs(denom) < 1e-12:          # ray parallel to this wall
            continue
        w = p - origin
        t = (w[0] * e[1] - w[1] * e[0]) / denom   # distance along the ray
        u = (w[0] * d[1] - w[1] * d[0]) / denom   # position along the segment (0..1)
        if t >= 0 and 0 <= u <= 1 and t < best:
            best = t
    return best


def lidar_scan(pose, segments, num_rays=360, max_range=10.0):
    """Simulate a 2D LiDAR at `pose = (x, y, theta)`.
    Returns (angles, ranges) in the *robot* frame; angle 0 is the robot's +x axis."""
    x, y, theta = pose
    angles = np.linspace(-np.pi, np.pi, num_rays, endpoint=False)
    ranges = np.array([cast_ray(np.array([x, y]), theta + a, segments, max_range) for a in angles])
    return angles, ranges


def polar_to_cartesian(angles, ranges):
    """Convert (angle, range) pairs to an (N, 2) array of (x, y) points."""
    return np.column_stack((ranges * np.cos(angles), ranges * np.sin(angles)))

In [ ]:
def best_rigid_transform(src, dst):
    """Closed-form (R, t) minimising sum ||R src_i + t - dst_i||^2 for paired points."""
    c_src, c_dst = src.mean(axis=0), dst.mean(axis=0)
    H = (src - c_src).T @ (dst - c_dst)
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:                 # guard against a reflection
        Vt[-1] *= -1
        R = Vt.T @ U.T
    t = c_dst - R @ c_src
    return R, t

def icp(src, dst, iterations=30, max_pair_distance=1.0):
    """Point-to-point ICP aligning `src` onto `dst`.
    Returns (R, t, history) where history[k] is the estimate after k iterations."""
    tree = KDTree(dst)
    R_total, t_total = np.eye(2), np.zeros(2)
    current = src.copy()
    history = [(R_total, t_total)]
    for _ in range(iterations):
        dist, idx = tree.query(current)
        keep = dist < max_pair_distance
        R, t = best_rigid_transform(current[keep], dst[idx[keep]])
        current = current @ R.T + t
        R_total, t_total = R @ R_total, R @ t_total + t
        history.append((R_total, t_total))
    return R_total, t_total, history

def report(name, R, t, true_pose):
    theta = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    print(f"{name:<22} estimated dx={t[0]:6.3f}  dy={t[1]:6.3f}  θ={theta:6.2f}°"
          f"   |  true dx={true_pose[0]:.3f}  dy={true_pose[1]:.3f}  θ={np.degrees(true_pose[2]):.2f}°")

## A featureless corridor

Two long parallel walls, much longer than the sensor's range so the robot
never sees the ends. The robot drives 0.5 m along the corridor and drifts
0.1 m sideways.

In [ ]:
corridor = [((-50, -1.5), (50, -1.5)),
            ((-50,  1.5), (50,  1.5))]

pose_0 = (0.0, 0.0, 0.0)
pose_1 = (0.5, 0.1, 0.0)
MAX_RANGE = 8.0

def scan_at(pose, segments):
    angles, ranges = lidar_scan(pose, segments, num_rays=360, max_range=MAX_RANGE)
    hit = ranges < MAX_RANGE                       # drop rays that saw nothing
    return polar_to_cartesian(angles[hit], ranges[hit])

ref = scan_at(pose_0, corridor)
new = scan_at(pose_1, corridor)

R, t, history = icp(new, ref)
report("corridor:", R, t, pose_1)

In [ ]:
aligned = new @ R.T + t
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 4))
for ax, pts, title in [(ax_l, new, "before ICP"), (ax_r, aligned, "after ICP")]:
    ax.scatter(ref[:, 0], ref[:, 1], s=6, label="reference (t0)")
    ax.scatter(pts[:, 0], pts[:, 1], s=6, label="new (t1)")
    ax.set_title(title); ax.axis("equal"); ax.legend(loc="upper right")
plt.show()

ICP recovers the sideways drift almost perfectly and reports essentially
**zero** forward motion. Visually the two scans look "aligned" — and in a
sense they are: a wall shifted along itself is still the same wall.

## Why: the cost is flat along the corridor

Evaluate ICP's own error — the mean nearest-neighbour distance — as we slide
the new scan along the corridor.

In [ ]:
def nn_cost(query, reference):
    return KDTree(reference).query(query)[0].mean()

dxs = np.linspace(-1.5, 2.0, 141)
cost_corridor = [nn_cost(new + np.array([dx, pose_1[1]]), ref) for dx in dxs]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dxs, cost_corridor, label="corridor")
ax.axvline(pose_1[0], color="r", ls="--", label="true dx")
ax.set_xlabel("candidate dx (m)"); ax.set_ylabel("mean NN distance (m)")
ax.set_title("Sliding along the corridor barely changes the cost"); ax.legend(); ax.grid(True)
plt.show()

Around the true value the curve is essentially flat. The only structure is
the laser's sampling pattern — small ripples where the new points land
exactly on top of reference points — plus a gentle rise at large `|dx|`
where the range-limited ends of the two scans stop overlapping. Neither
says anything about where along the corridor the robot actually is: that
direction is **unobservable** from the walls alone, and no matching
algorithm can recover it from this data. Point-to-point ICP simply latches
onto the nearest ripple, which, starting from `dx = 0`, is `dx ≈ 0`.

## One feature is enough

Add a single doorway alcove to one wall and run exactly the same code.

In [ ]:
doorway = [((-50, -1.5), (2.0, -1.5)),
           ((2.0, -1.5), (2.0, -2.5)), ((2.0, -2.5), (3.0, -2.5)), ((3.0, -2.5), (3.0, -1.5)),
           ((3.0, -1.5), (50, -1.5)),
           ((-50,  1.5), (50,  1.5))]

ref_d = scan_at(pose_0, doorway)
new_d = scan_at(pose_1, doorway)
R_d, t_d, history_d = icp(new_d, ref_d)
report("corridor + doorway:", R_d, t_d, pose_1)

cost_doorway = [nn_cost(new_d + np.array([dx, pose_1[1]]), ref_d) for dx in dxs]
print(f"lowest point-to-point cost is at dx = {dxs[np.argmin(cost_doorway)]:.3f}")

fig, (ax_map, ax_cost) = plt.subplots(1, 2, figsize=(14, 4))
ax_map.scatter(ref_d[:, 0], ref_d[:, 1], s=6, label="reference")
aligned_d = new_d @ R_d.T + t_d
ax_map.scatter(aligned_d[:, 0], aligned_d[:, 1], s=6, label="new, after ICP")
ax_map.axis("equal"); ax_map.legend(loc="upper right"); ax_map.set_title("With a doorway")

ax_cost.plot(dxs, cost_corridor, label="corridor only")
ax_cost.plot(dxs, cost_doorway, label="corridor + doorway")
ax_cost.axvline(pose_1[0], color="r", ls="--", label="true dx")
ax_cost.set_xlabel("candidate dx (m)"); ax_cost.set_ylabel("mean NN distance (m)")
ax_cost.legend(); ax_cost.grid(True); ax_cost.set_title("The doorway creates a minimum")
plt.show()

A handful of points on the doorway give the cost function a real minimum
at the true `dx`, and ICP now heads toward it — but *slowly*. After 30
iterations it is still well short of 0.5 m. Note how shallow that minimum
is compared with the ripples: only a few of the hundreds of points carry
any information about forward motion, while every wall point contributes
a sample-to-sample pull that fights the doorway points on each iteration.
Watch the estimate creep:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([t[0] for _, t in history], "o-", ms=3, label="corridor only")
ax.plot([t[0] for _, t in history_d], "s-", ms=3, label="corridor + doorway")
ax.axhline(pose_1[0], color="r", ls="--", label="true dx")
ax.set_xlabel("ICP iteration"); ax.set_ylabel("estimated dx (m)"); ax.legend(); ax.grid(True)
plt.show()

## What point-to-line changes

Point-to-point ICP pulls every new point toward a specific *sample* on the
reference wall. That is the wrong goal: the point should only be pulled
toward the wall's *surface*, and sliding along the surface should be free.

Point-to-line ICP makes that explicit. For each reference point it
estimates the wall's **normal** direction, and the error becomes the
distance along the normal only:

$$e_i = \mathbf{n}_i \cdot \big(\mathbf{q}_i - \mathbf{p}_i\big).$$

Two things follow:

* Wall points stop contributing spurious along-wall error, so the ripples
  in the cost curve above disappear.
* The remaining error is dominated by whatever features *do* constrain the
  motion (the doorway), so convergence toward the true `dx` is much faster.

What point-to-line does **not** do is make a truly featureless corridor
observable — nothing can. In that case a real SLAM system leans on odometry
or an IMU until a feature comes into view.

Below is a preview of the flattened cost: project each residual onto the
wall normal and recompute the curve.

In [ ]:
def estimate_normals(points, k=6):
    """Unit normal at each point from the principal direction of its k nearest neighbours."""
    _, idx = KDTree(points).query(points, k=k)
    normals = np.zeros_like(points)
    for i, nbrs in enumerate(idx):
        local = points[nbrs] - points[nbrs].mean(axis=0)
        _, _, Vt = np.linalg.svd(local)
        normals[i] = Vt[-1]                 # direction of least variance = normal
    return normals

def point_to_line_cost(query, reference, normals):
    dist, idx = KDTree(reference).query(query)
    residual = query - reference[idx]
    return np.abs((residual * normals[idx]).sum(axis=1)).mean()

normals_d = estimate_normals(ref_d)
cost_p2l = [point_to_line_cost(new_d + np.array([dx, pose_1[1]]), ref_d, normals_d) for dx in dxs]
print(f"lowest point-to-line cost is at dx = {dxs[np.argmin(cost_p2l)]:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dxs, cost_doorway, label="point-to-point")
ax.plot(dxs, cost_p2l, label="point-to-line")
ax.axvline(pose_1[0], color="r", ls="--", label="true dx")
ax.set_xlabel("candidate dx (m)"); ax.set_ylabel("mean error (m)")
ax.set_title("Corridor + doorway: point-to-line removes the sampling ripples"); ax.legend(); ax.grid(True)
plt.show()

## Summary

* In a featureless corridor, point-to-point ICP reports (almost) no
  forward motion: the along-wall direction is unobservable and the matcher
  locks onto sampling artefacts instead.
* A single feature restores a minimum, but point-to-point converges slowly
  because every wall point adds sample-matching noise.
* Point-to-line measures error only along the wall normal, removing that
  noise and letting the features dominate.

**Next:** the `5b` script implements normal estimation and the full
point-to-line ICP loop.